# Deep Learning Model: 3D CNN
This notebook implements a custom 3D Convolutional Neural Network for spatio-temporal sign language recognition.

## Imports and Environment Setup


In [5]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import albumentations as A
import json
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report, 
    precision_recall_fscore_support, average_precision_score
)
import pandas as pd

# Try to import mediapipe
try:
    import mediapipe as mp
    MEDIAPIPE_AVAILABLE = True
    print("✅ MediaPipe available - Hand detection enabled")
except ImportError:
    MEDIAPIPE_AVAILABLE = False
    print("⚠️  MediaPipe not available - Using full frame")
    print("   Install with: pip install mediapipe")

# Check PyTorch and CUDA
print(f"\n{'='*60}")
print("SYSTEM CONFIGURATION")
print(f"{'='*60}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"{'='*60}\n")

⚠️  MediaPipe not available - Using full frame
   Install with: pip install mediapipe

SYSTEM CONFIGURATION
PyTorch Version: 2.8.0+cu126
CUDA Available: True
CUDA Version: 12.6
GPU Device: Tesla T4
GPU Memory: 14.74 GB



# DEBUG TRACKER

In [6]:
class DebugTracker:
    def __init__(self):
        self.start_time = time.time()
    
    def log(self, msg):
        elapsed = time.time() - self.start_time
        print(f"🔹 [{elapsed:6.1f}s] {msg}")
    
    def success(self, msg):
        elapsed = time.time() - self.start_time
        print(f"✅ [{elapsed:6.1f}s] SUCCESS: {msg}")
    
    def error(self, msg):
        elapsed = time.time() - self.start_time
        print(f"❌ [{elapsed:6.1f}s] ERROR: {msg}")
    
    def warning(self, msg):
        elapsed = time.time() - self.start_time
        print(f"⚠️  [{elapsed:6.1f}s] WARNING: {msg}")

debug = DebugTracker()
debug.log("Debug tracker initialized")


🔹 [   0.0s] Debug tracker initialized


## Configuration
Define dataset paths, hyperparameters, and experiment settings.


In [7]:
class Config:
    # Data paths
    DATA_PATH = "/kaggle/input/asl-20-words-dataset-v1/Full Data"
    
    # Image and video parameters
    IMG_HEIGHT = 112  # Increased resolution for better accuracy
    IMG_WIDTH = 112
    NUM_FRAMES = 16  # Number of frames per video
    NUM_CLASSES = 20  # Number of sign language classes
    
    # Training parameters
    BATCH_SIZE = 8  # Smaller batch for better gradients
    EPOCHS = 30  # Maximum number of epochs
    LEARNING_RATE = 0.0005  # Initial learning rate
    
    # Device configuration
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Data loading
    MAX_VIDEOS_PER_CLASS = 70  # Limit videos per class
    NUM_WORKERS = 2 if torch.cuda.is_available() else 0
    
    # Features
    USE_AUGMENTATION = True
    USE_HAND_DETECTION = MEDIAPIPE_AVAILABLE
    
    # Model parameters
    DROPOUT_RATE = 0.5
    
    # Training strategies
    PATIENCE = 15  # Early stopping patience
    GRADIENT_CLIP = 1.0  # Gradient clipping max norm

config = Config()

# Display configuration
print(f"\n{'='*60}")
print("CONFIGURATION")
print(f"{'='*60}")
print(f"Device: {config.DEVICE}")
print(f"Image Size: {config.IMG_HEIGHT}x{config.IMG_WIDTH}")
print(f"Frames per Video: {config.NUM_FRAMES}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Max Videos/Class: {config.MAX_VIDEOS_PER_CLASS}")
print(f"Hand Detection: {'ENABLED 🖐️' if config.USE_HAND_DETECTION else 'DISABLED'}")
print(f"Data Augmentation: {'ENABLED' if config.USE_AUGMENTATION else 'DISABLED'}")
print(f"{'='*60}\n")


CONFIGURATION
Device: cuda
Image Size: 112x112
Frames per Video: 16
Batch Size: 8
Learning Rate: 0.0005
Max Videos/Class: 70
Hand Detection: DISABLED
Data Augmentation: ENABLED



## Hand Detector (MediaPipe)
Class for detecting the hand bounding box to extract the Region of Interest (ROI).


In [8]:
class HandDetector:
    def __init__(self, confidence=0.3):
        """
        Initialize MediaPipe hands detector
        
        Args:
            confidence: Detection/tracking confidence threshold
        """
        if not MEDIAPIPE_AVAILABLE:
            debug.warning("MediaPipe not available - Hand detection disabled")
            return
            
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=True,
            max_num_hands=2,  # Detect both hands
            min_detection_confidence=confidence,
            min_tracking_confidence=confidence
        )
        debug.success("Hand detector initialized")
    
    def detect_and_crop_hands(self, frame):
        """
        Detect hands and return cropped region containing both hands
        
        Args:
            frame: Input frame (BGR format)
            
        Returns:
            Cropped frame containing hand region(s) or original frame if no hands detected
        """
        if not MEDIAPIPE_AVAILABLE:
            return frame
        
        # Convert BGR to RGB for MediaPipe
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.hands.process(frame_rgb)
        
        # If no hands detected, return original frame
        if not results.multi_hand_landmarks:
            return frame
        
        h, w, _ = frame.shape
        hand_boxes = []
        
        # Process each detected hand
        for hand_landmarks in results.multi_hand_landmarks:
            # Get all landmark coordinates
            x_coords = [lm.x for lm in hand_landmarks.landmark]
            y_coords = [lm.y for lm in hand_landmarks.landmark]
            
            # Calculate bounding box
            x_min = int(min(x_coords) * w)
            x_max = int(max(x_coords) * w)
            y_min = int(min(y_coords) * h)
            y_max = int(max(y_coords) * h)
            
            # Add generous padding (50% on each side)
            padding_x = int((x_max - x_min) * 0.5)
            padding_y = int((y_max - y_min) * 0.5)
            
            x_min = max(0, x_min - padding_x)
            x_max = min(w, x_max + padding_x)
            y_min = max(0, y_min - padding_y)
            y_max = min(h, y_max + padding_y)
            
            hand_boxes.append((x_min, y_min, x_max, y_max))
        
        if not hand_boxes:
            return frame
        
        # Combine all hand boxes into one bounding box
        all_x_min = min(box[0] for box in hand_boxes)
        all_y_min = min(box[1] for box in hand_boxes)
        all_x_max = max(box[2] for box in hand_boxes)
        all_y_max = max(box[3] for box in hand_boxes)
        
        # Make it square for better aspect ratio
        box_w = all_x_max - all_x_min
        box_h = all_y_max - all_y_min
        box_size = max(box_w, box_h)
        
        # Center the square box
        center_x = (all_x_min + all_x_max) // 2
        center_y = (all_y_min + all_y_max) // 2
        
        x_min = max(0, center_x - box_size // 2)
        y_min = max(0, center_y - box_size // 2)
        x_max = min(w, center_x + box_size // 2)
        y_max = min(h, center_y + box_size // 2)
        
        # Crop hand region
        hand_roi = frame[y_min:y_max, x_min:x_max]
        
        # If crop is too small, return original
        if hand_roi.shape[0] < 50 or hand_roi.shape[1] < 50:
            return frame
        
        return hand_roi
    
    def __del__(self):
        """Clean up MediaPipe resources"""
        if MEDIAPIPE_AVAILABLE and hasattr(self, 'hands'):
            self.hands.close()

# Test hand detector if available
if config.USE_HAND_DETECTION:
    test_detector = HandDetector()
    debug.success("Hand detector test passed")
    del test_detector



## Video Processor
Handles ROI extraction, CLAHE contrast enhancement, and tensor normalization.


In [9]:
class ImprovedVideoProcessor:
    def __init__(self, target_size=(112, 112), num_frames=30, use_hand_detection=True):
        """
        Initialize video processor
        
        Args:
            target_size: Target frame size (width, height)
            num_frames: Number of frames to extract per video
            use_hand_detection: Whether to use hand detection
        """
        self.target_size = target_size
        self.num_frames = num_frames
        self.use_hand_detection = use_hand_detection and MEDIAPIPE_AVAILABLE
        
        # Initialize hand detector if available
        if self.use_hand_detection:
            self.hand_detector = HandDetector()
            debug.log("Video processor: Hand detection ENABLED")
        else:
            debug.log("Video processor: Hand detection DISABLED")
        
        # Data augmentation for training (using albumentations)
        self.train_transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.Rotate(limit=15, p=0.3),
        ])
        
        debug.success("Video processor initialized")
    
    def extract_frames_improved(self, video_path, augment=False):
        """
        Extract and preprocess frames from video
        
        Args:
            video_path: Path to video file
            augment: Whether to apply data augmentation
            
        Returns:
            Array of preprocessed frames or None if failed
        """
        try:
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                return None
            
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames <= 0:
                cap.release()
                return None
            
            # Uniform frame sampling across video
            frame_indices = np.linspace(0, total_frames-1, self.num_frames, dtype=int)
            
            frames = []
            for idx in frame_indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                
                if ret and frame is not None:
                    # Apply hand detection if enabled
                    if self.use_hand_detection:
                        frame = self.hand_detector.detect_and_crop_hands(frame)
                    
                    # Convert to grayscale
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    
                    # Apply augmentation if training
                    if augment and config.USE_AUGMENTATION:
                        augmented = self.train_transform(image=gray)
                        gray = augmented['image']
                    
                    # Resize to target size
                    resized = cv2.resize(gray, self.target_size)
                    
                    # CLAHE for better contrast
                    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                    enhanced = clahe.apply(resized)
                    
                    # Normalize to [0, 1]
                    normalized = enhanced.astype(np.float32) / 255.0
                    
                    # Standardize (zero mean, unit variance)
                    normalized = (normalized - 0.5) / 0.5
                    
                    frames.append(normalized)
                else:
                    # Fill missing frames with zeros
                    frames.append(np.zeros(self.target_size, dtype=np.float32))
            
            cap.release()
            
            # Ensure correct number of frames
            while len(frames) < self.num_frames:
                frames.append(np.zeros(self.target_size, dtype=np.float32))
            
            return np.array(frames[:self.num_frames])
            
        except Exception as e:
            debug.error(f"Failed to process video: {str(e)}")
            return None

# Test video processor
processor_test = ImprovedVideoProcessor(
    target_size=(config.IMG_WIDTH, config.IMG_HEIGHT),
    num_frames=config.NUM_FRAMES,
    use_hand_detection=config.USE_HAND_DETECTION
)
debug.success("Video processor test passed")


🔹 [   0.1s] Video processor: Hand detection DISABLED
✅ [   0.1s] SUCCESS: Video processor initialized
✅ [   0.1s] SUCCESS: Video processor test passed


/tmp/ipykernel_55/133486404.py:25: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),


## Model Architecture: 3D-CNN
The core deep learning model architecture.


In [10]:
class Improved3DCNN(nn.Module):
    def __init__(self, num_classes=20, dropout_rate=0.5):
        """
        Initialize 3D CNN model
        
        Architecture:
        - 4 Convolutional blocks with increasing filters
        - Each block: Conv3D → BatchNorm → ReLU → Conv3D → BatchNorm → ReLU → MaxPool
        - Global average pooling
        - 3 Fully connected layers with dropout
        
        Args:
            num_classes: Number of output classes
            dropout_rate: Dropout probability
        """
        super(Improved3DCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Block 1: 1 → 64 channels
            nn.Conv3d(1, 64, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.Conv3d(64, 64, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 2, 2)),  # Temporal: 30, Spatial: 56x56
            
            # Block 2: 64 → 128 channels
            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.Conv3d(128, 128, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((2, 2, 2)),  # Temporal: 15, Spatial: 28x28
            
            # Block 3: 128 → 256 channels
            nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            nn.Conv3d(256, 256, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((2, 2, 2)),  # Temporal: 7, Spatial: 14x14
            
            # Block 4: 256 → 512 channels
            nn.Conv3d(256, 512, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(512),
            nn.ReLU(inplace=True),
            nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 4, 4))  # Global pooling
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(512 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, num_classes)
        )
        
        # Initialize weights
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize model weights using Kaiming initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm3d) or isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input tensor (batch, channels, frames, height, width)
            
        Returns:
            Class logits (batch, num_classes)
        """
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

# Create and display model
model = Improved3DCNN(num_classes=config.NUM_CLASSES, dropout_rate=config.DROPOUT_RATE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*60}")
print("MODEL ARCHITECTURE")
print(f"{'='*60}")
print(model)
print(f"\n{'='*60}")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Model Size: {total_params * 4 / (1024**2):.2f} MB (float32)")
print(f"{'='*60}\n")

del model  # Free memory



MODEL ARCHITECTURE
Improved3DCNN(
  (features): Sequential(
    (0): Conv3d(1, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
    (7): Conv3d(64, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (8): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv3d(128, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (11): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool3d(kernel_s

## Dataset Loader
PyTorch dataset class to load and process video sequences.


In [11]:
class ImprovedDataset(Dataset):
    def __init__(self, file_paths, labels, processor, preprocess_now=True, augment=False):
        """
        Initialize dataset
        
        Args:
            file_paths: List of video file paths
            labels: List of corresponding labels
            processor: Video processor instance
            preprocess_now: Whether to preprocess all videos immediately
            augment: Whether to apply augmentation
        """
        self.file_paths = file_paths
        self.labels = labels
        self.processor = processor
        self.cache = {}
        self.augment = augment
        
        if preprocess_now:
            debug.log(f"Preprocessing {len(file_paths)} videos (augment={augment})...")
            self._preprocess_all()
    
    def _preprocess_all(self):
        """Preprocess all videos upfront and cache them"""
        valid_indices = []
        failed_count = 0
        
        for i, path in enumerate(tqdm(self.file_paths, desc="Processing videos")):
            frames = self.processor.extract_frames_improved(path, augment=False)
            if frames is not None:
                self.cache[i] = frames
                valid_indices.append(i)
            else:
                failed_count += 1
        
        # Keep only valid videos
        self.file_paths = [self.file_paths[i] for i in valid_indices]
        self.labels = [self.labels[i] for i in valid_indices]
        
        debug.success(f"Kept {len(self.file_paths)} valid videos ({failed_count} failed)")
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        """
        Get a single sample
        
        Returns:
            tuple: (frames_tensor, label)
        """
        # Get cached frames or process on-the-fly
        frames = self.cache.get(idx)
        if frames is None:
            frames = self.processor.extract_frames_improved(
                self.file_paths[idx], 
                augment=self.augment
            )
            if frames is None:
                frames = np.zeros((config.NUM_FRAMES, config.IMG_HEIGHT, config.IMG_WIDTH))
        
        # Apply temporal augmentation during training
        if self.augment and np.random.random() > 0.5:
            # Random temporal shift
            shift = np.random.randint(-2, 3)
            frames = np.roll(frames, shift, axis=0)
        
        # Convert to tensor: (frames, height, width) → (1, frames, height, width)
        frames_tensor = torch.FloatTensor(frames).unsqueeze(0)
        
        return frames_tensor, self.labels[idx]

debug.success("Dataset class defined")

✅ [   0.4s] SUCCESS: Dataset class defined


# DATA SETUP FUNCTION

In [12]:
def save_class_mapping(classes, filename='class_mapping.json'):
    """Save class index to name mapping"""
    class_mapping = {i: class_name for i, class_name in enumerate(classes)}
    with open(filename, 'w') as f:
        json.dump(class_mapping, f, indent=2)
    debug.success(f"Class mapping saved: {len(classes)} classes")
    return class_mapping

def setup_data():
    debug.log("Setting up data...")
    
    classes = sorted([d for d in os.listdir(config.DATA_PATH) 
                     if os.path.isdir(os.path.join(config.DATA_PATH, d)) and d != 'README.md'])
    
    if len(classes) == 0:
        debug.error(f"No class directories found in {config.DATA_PATH}")
        return None, None, None, None, None  # Added None for classes
    
    class_mapping = save_class_mapping(classes)
    
    file_paths = []
    labels = []
    
    for idx, class_name in enumerate(classes):
        class_path = os.path.join(config.DATA_PATH, class_name)
        videos = [os.path.join(class_path, f) for f in os.listdir(class_path)[:config.MAX_VIDEOS_PER_CLASS]
                 if f.lower().endswith(('.mp4', '.avi', '.mov'))]
        
        file_paths.extend(videos)
        labels.extend([idx] * len(videos))
        debug.log(f"{class_name}: {len(videos)} videos")
    
    debug.success(f"Total: {len(file_paths)} videos across {len(classes)} classes")
    
    train_files, val_files, train_labels, val_labels = train_test_split(
        file_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    debug.log(f"Train: {len(train_files)} videos, Val: {len(val_files)} videos")
    
    return train_files, val_files, train_labels, val_labels, classes 


## Evaluation Metrics
Classes and functions for computing accuracy, precision, recall, and plotting confusion matrices.


In [13]:
class ComprehensiveMetrics:
    """Calculate all evaluation metrics for sign language recognition"""
    
    def __init__(self, num_classes, class_names=None):
        self.num_classes = num_classes
        self.class_names = class_names or [f"Class_{i}" for i in range(num_classes)]
        self.reset()
    
    def reset(self):
        self.all_predictions = []
        self.all_targets = []
        self.all_probabilities = []
        self.inference_times = []
    
    def update(self, predictions, targets, probabilities=None, inference_time=None):
        self.all_predictions.extend(predictions.cpu().numpy())
        self.all_targets.extend(targets.cpu().numpy())
        if probabilities is not None:
            self.all_probabilities.extend(probabilities.cpu().numpy())
        if inference_time is not None:
            self.inference_times.append(inference_time)
    
    def compute_all_metrics(self):
        """Compute all metrics at once"""
        preds = np.array(self.all_predictions)
        targets = np.array(self.all_targets)
        
        # Basic accuracy
        accuracy = 100 * (preds == targets).sum() / len(targets)
        
        # Per-class metrics
        precision, recall, f1, support = precision_recall_fscore_support(
            targets, preds, average=None, zero_division=0
        )
        
        # Averages
        p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
            targets, preds, average='macro', zero_division=0
        )
        p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
            targets, preds, average='weighted', zero_division=0
        )
        
        # Top-K accuracy
        top3_acc = top5_acc = map_score = None
        if self.all_probabilities:
            probs = np.array(self.all_probabilities)
            
            # Top-3
            top3_preds = np.argsort(probs, axis=1)[:, -3:]
            top3_acc = 100 * sum(targets[i] in top3_preds[i] for i in range(len(targets))) / len(targets)
            
            # Top-5
            top5_preds = np.argsort(probs, axis=1)[:, -5:]
            top5_acc = 100 * sum(targets[i] in top5_preds[i] for i in range(len(targets))) / len(targets)
            
            # mAP
            targets_onehot = np.eye(self.num_classes)[targets]
            aps = []
            for i in range(self.num_classes):
                try:
                    ap = average_precision_score(targets_onehot[:, i], probs[:, i])
                    aps.append(ap)
                except:
                    aps.append(0.0)
            map_score = np.mean(aps) * 100
        
        # Inference metrics
        fps = latency = None
        if self.inference_times:
            times = np.array(self.inference_times)
            latency = np.mean(times) * 1000  # ms
            fps = 1.0 / np.mean(times) if np.mean(times) > 0 else 0
        
        # Per-class accuracy
        cm = confusion_matrix(targets, preds)
        per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
        
        return {
            'accuracy': accuracy,
            'precision_per_class': precision * 100,
            'recall_per_class': recall * 100,
            'f1_per_class': f1 * 100,
            'support': support,
            'precision_macro': p_macro * 100,
            'recall_macro': r_macro * 100,
            'f1_macro': f1_macro * 100,
            'precision_weighted': p_weighted * 100,
            'recall_weighted': r_weighted * 100,
            'f1_weighted': f1_weighted * 100,
            'top3_accuracy': top3_acc,
            'top5_accuracy': top5_acc,
            'map': map_score,
            'eerr': accuracy,  # Same as accuracy for this task
            'fps': fps,
            'latency_ms': latency,
            'confusion_matrix': cm,
            'per_class_acc': {self.class_names[i]: acc for i, acc in enumerate(per_class_acc)}
        }
    
    def get_classification_report(self):
        preds = np.array(self.all_predictions)
        targets = np.array(self.all_targets)
        return classification_report(targets, preds, target_names=self.class_names, digits=4, zero_division=0)

debug.success("Comprehensive metrics calculator defined")

✅ [   0.6s] SUCCESS: Comprehensive metrics calculator defined


# VISUALIZATION FUNCTIONS 

In [14]:
def create_all_visualizations(metrics_dict, metrics_calc, epoch='final'):
    """Create all visualization plots"""
    
    # 1. Confusion Matrix
    plt.figure(figsize=(14, 12))
    cm_norm = metrics_dict['confusion_matrix'].astype('float') / metrics_dict['confusion_matrix'].sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=metrics_calc.class_names, yticklabels=metrics_calc.class_names)
    plt.title('Confusion Matrix (Normalized)', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted', fontweight='bold')
    plt.ylabel('True', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{epoch}.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # 2. Per-Class Metrics
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    classes = metrics_calc.class_names
    
    axes[0, 0].barh(classes, metrics_dict['precision_per_class'], color='steelblue')
    axes[0, 0].set_xlabel('Precision (%)', fontweight='bold')
    axes[0, 0].set_title('Per-Class Precision', fontweight='bold')
    axes[0, 0].grid(axis='x', alpha=0.3)
    
    axes[0, 1].barh(classes, metrics_dict['recall_per_class'], color='coral')
    axes[0, 1].set_xlabel('Recall (%)', fontweight='bold')
    axes[0, 1].set_title('Per-Class Recall', fontweight='bold')
    axes[0, 1].grid(axis='x', alpha=0.3)
    
    axes[1, 0].barh(classes, metrics_dict['f1_per_class'], color='mediumseagreen')
    axes[1, 0].set_xlabel('F1-Score (%)', fontweight='bold')
    axes[1, 0].set_title('Per-Class F1-Score', fontweight='bold')
    axes[1, 0].grid(axis='x', alpha=0.3)
    
    axes[1, 1].barh(classes, metrics_dict['support'], color='plum')
    axes[1, 1].set_xlabel('Samples', fontweight='bold')
    axes[1, 1].set_title('Per-Class Support', fontweight='bold')
    axes[1, 1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'per_class_metrics_{epoch}.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # 3. Summary Dashboard
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)
    
    # Overall metrics
    ax1 = fig.add_subplot(gs[0, :])
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    values = [metrics_dict['accuracy'], metrics_dict['precision_weighted'], 
              metrics_dict['recall_weighted'], metrics_dict['f1_weighted']]
    colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']
    bars = ax1.bar(metric_names, values, color=colors, alpha=0.8, edgecolor='black')
    ax1.set_ylabel('Score (%)', fontweight='bold')
    ax1.set_title('Overall Metrics', fontweight='bold', fontsize=14)
    ax1.set_ylim([0, 105])
    ax1.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%', ha='center', va='bottom', fontweight='bold')
    
    # Top-K
    ax2 = fig.add_subplot(gs[1, 0])
    if metrics_dict['top3_accuracy']:
        topk_vals = [metrics_dict['accuracy'], metrics_dict['top3_accuracy'], metrics_dict['top5_accuracy']]
        ax2.plot(['Top-1', 'Top-3', 'Top-5'], topk_vals, 'o-', linewidth=2, markersize=10, color='#E91E63')
        ax2.set_ylabel('Accuracy (%)', fontweight='bold')
        ax2.set_title('Top-K Accuracy', fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.set_ylim([0, 105])
    
    # Advanced metrics
    ax3 = fig.add_subplot(gs[1, 1])
    if metrics_dict['map']:
        bars = ax3.bar(['mAP', 'EERR'], [metrics_dict['map'], metrics_dict['eerr']], 
                      color=['#FF5722', '#00BCD4'], alpha=0.8)
        ax3.set_ylabel('Score (%)', fontweight='bold')
        ax3.set_title('Advanced Metrics', fontweight='bold')
        ax3.set_ylim([0, 105])
        ax3.grid(axis='y', alpha=0.3)
        for bar in bars:
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}%', ha='center', va='bottom', fontweight='bold')
    
    # Inference
    ax4 = fig.add_subplot(gs[1, 2])
    if metrics_dict['fps']:
        ax4.text(0.5, 0.7, f"{metrics_dict['fps']:.2f}", ha='center', va='center', 
                fontsize=36, fontweight='bold', color='#4CAF50')
        ax4.text(0.5, 0.5, "FPS", ha='center', va='center', fontsize=14, fontweight='bold')
        ax4.text(0.5, 0.3, f"Latency: {metrics_dict['latency_ms']:.2f}ms", 
                ha='center', va='center', fontsize=12, color='#666')
        ax4.set_xlim([0, 1])
        ax4.set_ylim([0, 1])
        ax4.axis('off')
        ax4.set_title('Inference Speed', fontweight='bold')
    
    plt.savefig(f'metrics_summary_{epoch}.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    debug.success(f"All visualizations saved for epoch {epoch}")


def print_comprehensive_report(metrics_dict, classification_report_str):
    """Print detailed metrics report"""
    print("\n" + "="*80)
    print("📊 COMPREHENSIVE EVALUATION METRICS")
    print("="*80)
    
    print("\n🎯 OVERALL METRICS:")
    print(f"  • Accuracy:               {metrics_dict['accuracy']:.2f}%")
    print(f"  • Precision (Weighted):   {metrics_dict['precision_weighted']:.2f}%")
    print(f"  • Recall (Weighted):      {metrics_dict['recall_weighted']:.2f}%")
    print(f"  • F1-Score (Weighted):    {metrics_dict['f1_weighted']:.2f}%")
    
    print("\n🏆 TOP-K ACCURACY:")
    print(f"  • Top-1 (Standard):       {metrics_dict['accuracy']:.2f}%")
    if metrics_dict['top3_accuracy']:
        print(f"  • Top-3:                  {metrics_dict['top3_accuracy']:.2f}%")
        print(f"  • Top-5:                  {metrics_dict['top5_accuracy']:.2f}%")
    
    print("\n📈 ADVANCED METRICS:")
    if metrics_dict['map']:
        print(f"  • Mean Avg Precision:     {metrics_dict['map']:.2f}%")
    print(f"  • End-to-End Recognition: {metrics_dict['eerr']:.2f}%")
    
    print("\n📊 AVERAGING METHODS:")
    print(f"  • Macro Avg  - P: {metrics_dict['precision_macro']:.2f}% | R: {metrics_dict['recall_macro']:.2f}% | F1: {metrics_dict['f1_macro']:.2f}%")
    print(f"  • Weighted   - P: {metrics_dict['precision_weighted']:.2f}% | R: {metrics_dict['recall_weighted']:.2f}% | F1: {metrics_dict['f1_weighted']:.2f}%")
    
    if metrics_dict['fps']:
        print("\n⚡ INFERENCE PERFORMANCE:")
        print(f"  • FPS:                    {metrics_dict['fps']:.2f}")
        print(f"  • Latency:                {metrics_dict['latency_ms']:.2f} ms")
    
    print("\n🎨 PER-CLASS ACCURACY:")
    for class_name, acc in sorted(metrics_dict['per_class_acc'].items(), key=lambda x: x[1], reverse=True):
        print(f"  • {class_name:<25} {acc:6.2f}%")
    
    print("\n" + "="*80)
    print("DETAILED CLASSIFICATION REPORT")
    print("="*80)
    print(classification_report_str)
    print("="*80 + "\n")

debug.success("Visualization functions defined")

✅ [   0.6s] SUCCESS: Visualization functions defined


# EVALUATION FUNCTION

In [15]:
def run_comprehensive_evaluation(model, data_loader, device, num_classes, class_names):
    """Run full evaluation with all metrics"""
    model.eval()
    metrics_calc = ComprehensiveMetrics(num_classes, class_names)
    
    print("\n🔍 Running comprehensive evaluation...")
    
    with torch.no_grad():
        for data, targets in tqdm(data_loader, desc="Evaluating"):
            data, targets = data.to(device), targets.to(device)
            
            start_time = time.time()
            outputs = model(data)
            inference_time = (time.time() - start_time) / len(data)
            
            probabilities = torch.softmax(outputs, dim=1)
            _, predictions = torch.max(outputs, 1)
            
            metrics_calc.update(predictions, targets, probabilities, inference_time)
    
    # Compute all metrics
    metrics_dict = metrics_calc.compute_all_metrics()
    classification_rep = metrics_calc.get_classification_report()
    
    return metrics_dict, classification_rep, metrics_calc

debug.success("Evaluation function defined")


✅ [   0.6s] SUCCESS: Evaluation function defined


## Training Loop
Function implementing the main training loop with early stopping and checkpointing.


In [16]:
def improved_training():
    """
    Train the 3D CNN model with improved techniques
    
    Returns:
        tuple: (best_accuracy, final_accuracy)
    """
    train_files, val_files, train_labels, val_labels, classes = setup_data()  # Added classes
    
    if train_files is None:
        debug.error("Data setup failed!")
        return 0, 0
    
    # Create video processor
    processor = ImprovedVideoProcessor(
        target_size=(config.IMG_WIDTH, config.IMG_HEIGHT),
        num_frames=config.NUM_FRAMES,
        use_hand_detection=config.USE_HAND_DETECTION
    )
    
    # Create datasets
    debug.log("Creating training dataset with augmentation...")
    train_dataset = ImprovedDataset(
        train_files, train_labels, processor, 
        preprocess_now=True, augment=True
    )
    
    debug.log("Creating validation dataset...")
    val_dataset = ImprovedDataset(
        val_files, val_labels, processor, 
        preprocess_now=True, augment=False
    )
    
    if len(train_dataset) == 0:
        debug.error("No valid training videos!")
        return 0, 0
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=True, 
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=False, 
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    
    # Initialize model
    debug.log("Initializing model...")
    model = Improved3DCNN(num_classes=config.NUM_CLASSES, dropout_rate=config.DROPOUT_RATE)
    model.to(config.DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    debug.success(f"Model initialized: {total_params:,} parameters")
    
    # Loss function with label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # Optimizer: AdamW with weight decay
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config.LEARNING_RATE, 
        weight_decay=0.01, 
        betas=(0.9, 0.999)
    )
    
    # Learning rate scheduler: Cosine annealing with warm restarts
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )
    
    # Training tracking
    best_acc = 0
    patience_counter = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    debug.log("Starting training...")
    print(f"\n{'='*60}")
    print("TRAINING PROGRESS")
    print(f"{'='*60}\n")
    
    for epoch in range(config.EPOCHS):
        epoch_start = time.time()
        
        # ==================== TRAINING PHASE ====================
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        
        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.EPOCHS} [TRAIN]')
        for data, targets in train_bar:
            data, targets = data.to(config.DEVICE), targets.to(config.DEVICE)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.GRADIENT_CLIP)
            
            # Update weights
            optimizer.step()
            
            # Track metrics
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()
            
            # Update progress bar
            train_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * train_correct / train_total:.2f}%'
            })
        
        # Update learning rate
        scheduler.step()
        
        # ==================== VALIDATION PHASE ====================
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{config.EPOCHS} [VAL]  ')
            for data, targets in val_bar:
                data, targets = data.to(config.DEVICE), targets.to(config.DEVICE)
                
                outputs = model(data)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()
                
                val_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100 * val_correct / val_total:.2f}%'
                })
        
        # Calculate epoch metrics
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_acc': best_acc,
                'config': vars(config)
            }, 'improved_best_model.pth')
            patience_counter = 0
            debug.success(f"New best model saved! Val Acc: {val_acc:.2f}%")
        else:
            patience_counter += 1
        
        # Epoch summary
        epoch_time = time.time() - epoch_start
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{config.EPOCHS} Summary ({epoch_time:.1f}s)")
        print(f"{'='*60}")
        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:5.2f}%")
        print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:5.2f}%")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        print(f"Best Val Acc: {best_acc:.2f}% (patience: {patience_counter}/{config.PATIENCE})")
        print(f"{'='*60}\n")
        
        # Early stopping
        if patience_counter >= config.PATIENCE:
            debug.success(f"Early stopping triggered at epoch {epoch+1}")
            break
    
    print("\n" + "="*80)
    print("🎯 FINAL MODEL EVALUATION WITH ALL METRICS")
    print("="*80)
    
    # Load best model
    checkpoint = torch.load('improved_best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(config.DEVICE)
    
    # Run comprehensive evaluation
    final_metrics, classification_rep, metrics_calc = run_comprehensive_evaluation(
        model, val_loader, config.DEVICE, config.NUM_CLASSES, classes
    )
    
    # Print detailed report
    print_comprehensive_report(final_metrics, classification_rep)
    
    # Create all visualizations
    create_all_visualizations(final_metrics, metrics_calc, epoch='final')
    
    # Save metrics to JSON
    # Convert numpy arrays to lists for JSON serialization
    metrics_to_save = {k: (v.tolist() if isinstance(v, np.ndarray) else v) 
                       for k, v in final_metrics.items() if k != 'confusion_matrix'}
    
    with open('final_metrics.json', 'w') as f:
        json.dump(metrics_to_save, f, indent=2)
    
    debug.success("Comprehensive evaluation complete!")
    debug.success("All metrics and visualizations saved!")
    
    print("\n📁 GENERATED FILES:")
    print("  ✓ confusion_matrix_final.png")
    print("  ✓ per_class_metrics_final.png")
    print("  ✓ metrics_summary_final.png")
    print("  ✓ final_metrics.json")
    print("  ✓ improved_best_model.pth")
    print("  ✓ class_mapping.json")
    print("  ✓ improved_training.png")
    
    return best_acc, val_accs[-1] if val_accs else 0

## Main Execution (Results)
Run the training pipeline and display the results. **(Outputs Preserved)**


In [17]:
def main():
    """Main function to run the complete pipeline"""
    print("\n" + "="*60)
    print("🚀 3D CNN FOR ARABIC SIGN LANGUAGE RECOGNITION")
    print("="*60)
    print("\n📋 IMPROVEMENTS:")
    print("  • Higher resolution (112x112)")
    print("  • More frames (30) for better temporal modeling")
    print("  • Deeper 3D CNN architecture")
    print("  • Data augmentation (brightness, contrast, rotation)")
    print("  • CLAHE preprocessing for enhanced contrast")
    print("  • Label smoothing and gradient clipping")
    print("  • Cosine annealing with warm restarts")
    print("  • Early stopping with patience")
    if config.USE_HAND_DETECTION:
        print("  • 🖐️ Hand detection with MediaPipe (ENABLED)")
    else:
        print("  • Hand detection (DISABLED)")
    print("="*60 + "\n")
    
    # Run training
    best_acc, final_acc = improved_training()
    
    # Final summary
    total_time = time.time() - debug.start_time
    
    print("\n" + "="*60)
    print("🎯 FINAL RESULTS")
    print("="*60)
    print(f"Best Validation Accuracy:    {best_acc:6.2f}%")
    print(f"Final Validation Accuracy:   {final_acc:6.2f}%")
    print(f"Total Training Time:         {total_time/60:6.1f} minutes")
    print(f"Model Parameters:            {sum(p.numel() for p in Improved3DCNN().parameters()):,}")
    if config.USE_HAND_DETECTION:
        print(f"Hand Detection:              ENABLED 🖐️")
    else:
        print(f"Hand Detection:              DISABLED")
    print("="*60)
    
    # Performance assessment
    if best_acc >= 80:
        print("\n🎉 EXCELLENT! Model achieved high accuracy!")
    elif best_acc >= 70:
        print("\n✅ GOOD! Model performs well!")
    elif best_acc >= 60:
        print("\n📈 DECENT! Room for improvement")
    elif best_acc >= 50:
        print("\n⚠️  FAIR! Consider tuning hyperparameters")
    else:
        print("\n❌ NEEDS WORK! Check data quality and model architecture")
    
    # Save all files and show instructions
    print("\n" + "="*60)
    print("📁 OUTPUT FILES SAVED")
    print("="*60)
    
    import os
    current_dir = os.getcwd()
    
    files_info = [
        ('improved_best_model.pth', 'Best model weights'),
        ('class_mapping.json', 'Class index to name mapping'),
        ('improved_training.png', 'Training visualization')
    ]
    
    print(f"\n✅ All files saved in: {current_dir}")
    print("\n📄 Files created:")
    for filename, description in files_info:
        if os.path.exists(filename):
            file_size = os.path.getsize(filename) / (1024**2)  # Size in MB
            print(f"  ✓ {filename}")
            print(f"    - {description}")
            print(f"    - Size: {file_size:.2f} MB")
        else:
            print(f"  ✗ {filename} - NOT FOUND")
    
    print("\n" + "="*60)
    print("📥 HOW TO DOWNLOAD FROM KAGGLE")
    print("="*60)
    print("\n1️⃣  Wait for notebook to finish running completely")
    print("2️⃣  Look at the RIGHT SIDEBAR")
    print("3️⃣  Click on 'Output' tab (folder icon)")
    print("4️⃣  You'll see all saved files listed there")
    print("5️⃣  Click the three dots (...) next to each file")
    print("6️⃣  Select 'Download' to save to your computer")
    print("\n💡 TIP: You can also download all files at once:")
    print("   - Click 'Save Version' → Wait for completion")
    print("   - Go to 'Output' section of your notebook version")
    print("   - Download all files as a zip")
    print("="*60 + "\n")
    
    debug.success("Training completed successfully!")
    debug.log("All output files are ready for download!")

# Run if executing as main script
if __name__ == "__main__":
    main()



🚀 3D CNN FOR ARABIC SIGN LANGUAGE RECOGNITION

📋 IMPROVEMENTS:
  • Higher resolution (112x112)
  • More frames (30) for better temporal modeling
  • Deeper 3D CNN architecture
  • Data augmentation (brightness, contrast, rotation)
  • CLAHE preprocessing for enhanced contrast
  • Label smoothing and gradient clipping
  • Cosine annealing with warm restarts
  • Early stopping with patience
  • Hand detection (DISABLED)

🔹 [   0.7s] Setting up data...
✅ [   0.7s] SUCCESS: Class mapping saved: 20 classes
🔹 [   0.7s] baby: 70 videos
🔹 [   0.7s] eat: 70 videos
🔹 [   0.7s] father: 70 videos
🔹 [   0.7s] finish: 70 videos
🔹 [   0.7s] good: 70 videos
🔹 [   0.7s] happy: 70 videos
🔹 [   0.8s] hear: 70 videos
🔹 [   0.8s] house: 70 videos
🔹 [   0.8s] important: 70 videos
🔹 [   0.8s] love: 70 videos
🔹 [   0.8s] mall: 70 videos
🔹 [   0.8s] me: 70 videos
🔹 [   0.8s] mosque: 70 videos
🔹 [   0.8s] mother: 70 videos
🔹 [   0.9s] normal: 70 videos
🔹 [   0.9s] sad: 70 videos


/tmp/ipykernel_55/133486404.py:25: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),


🔹 [   0.9s] stop: 70 videos
🔹 [   0.9s] thanks: 70 videos
🔹 [   0.9s] thinking: 70 videos
🔹 [   0.9s] worry: 70 videos
✅ [   0.9s] SUCCESS: Total: 1400 videos across 20 classes
🔹 [   0.9s] Train: 1120 videos, Val: 280 videos
🔹 [   0.9s] Video processor: Hand detection DISABLED
✅ [   0.9s] SUCCESS: Video processor initialized
🔹 [   0.9s] Creating training dataset with augmentation...
🔹 [   0.9s] Preprocessing 1120 videos (augment=True)...


Processing videos:   5%|▌         | 58/1120 [01:29<32:10,  1.82s/it][h264 @ 0xd420b80] Invalid NAL unit size (8270 > 5705).
[h264 @ 0xd420b80] Error splitting the input into NAL units.
[h264 @ 0xd420b80] Invalid NAL unit size (8270 > 5705).
[h264 @ 0xd420b80] Error splitting the input into NAL units.
[h264 @ 0xd2857c0] Invalid NAL unit size (8270 > 5705).
[h264 @ 0xd2857c0] Error splitting the input into NAL units.
[h264 @ 0xd2857c0] Invalid NAL unit size (8270 > 5705).
[h264 @ 0xd2857c0] Error splitting the input into NAL units.
[h264 @ 0xd2f6f80] Invalid NAL unit size (8270 > 5705).
[h264 @ 0xd2f6f80] Error splitting the input into NAL units.
[h264 @ 0xd2f6f80] Invalid NAL unit size (8270 > 5705).
[h264 @ 0xd2f6f80] Error splitting the input into NAL units.
Processing videos:   7%|▋         | 82/1120 [02:11<24:31,  1.42s/it][h264 @ 0xd285740] Invalid NAL unit size (11505 > 10359).
[h264 @ 0xd285740] Error splitting the input into NAL units.
Processing videos:   7%|▋         | 83/1120

✅ [1806.4s] SUCCESS: Kept 1120 valid videos (0 failed)
🔹 [1806.4s] Creating validation dataset...
🔹 [1806.4s] Preprocessing 280 videos (augment=False)...


...


Epoch 29/30 Summary (224.1s)
Train Loss: 0.8718 | Train Acc: 93.57%
Val Loss:   1.0096 | Val Acc:   85.71%
Learning Rate: 0.000004
Best Val Acc: 87.14% (patience: 3/15)



Epoch 30/30 [VAL]  : 100%|██████████| 35/35 [00:15<00:00,  2.29it/s, loss=0.9384, acc=86.43%]



Epoch 30/30 Summary (224.4s)
Train Loss: 0.8886 | Train Acc: 92.95%
Val Loss:   0.9658 | Val Acc:   86.43%
Learning Rate: 0.000500
Best Val Acc: 87.14% (patience: 4/15)


🎯 FINAL MODEL EVALUATION WITH ALL METRICS

🔍 Running comprehensive evaluation...


Evaluating: 100%|██████████| 35/35 [00:15<00:00,  2.29it/s]



📊 COMPREHENSIVE EVALUATION METRICS

🎯 OVERALL METRICS:
  • Accuracy:               87.14%
  • Precision (Weighted):   88.28%
  • Recall (Weighted):      87.14%
  • F1-Score (Weighted):    87.33%

🏆 TOP-K ACCURACY:
  • Top-1 (Standard):       87.14%
  • Top-3:                  98.93%
  • Top-5:                  99.29%

📈 ADVANCED METRICS:
  • Mean Avg Precision:     92.25%
  • End-to-End Recognition: 87.14%

📊 AVERAGING METHODS:
  • Macro Avg  - P: 88.28% | R: 87.14% | F1: 87.33%
  • Weighted   - P: 88.28% | R: 87.14% | F1: 87.33%

⚡ INFERENCE PERFORMANCE:
  • FPS:                    3133.88
  • Latency:                0.32 ms

🎨 PER-CLASS ACCURACY:
  • baby                      100.00%
  • house                     100.00%
  • father                     92.86%
  • happy                      92.86%
  • hear                       92.86%
  • normal                     92.86%
  • stop                       92.86%
  • thanks                     92.86%
  • thinking                   92.86%


In [18]:
"""
This cell contains analysis of potential issues in the code
Run this to see recommendations and potential improvements
"""

def analyze_code_issues():
    """Analyze potential issues and provide recommendations"""
    print("\n" + "="*60)
    print("🔍 CODE ANALYSIS & RECOMMENDATIONS")
    print("="*60 + "\n")
    
    issues = []
    recommendations = []
    
    # Check MediaPipe availability
    if not MEDIAPIPE_AVAILABLE:
        issues.append("MediaPipe not installed - hand detection disabled")
        recommendations.append("Install MediaPipe: pip install mediapipe")
    
    # Check CUDA
    if not torch.cuda.is_available():
        issues.append("CUDA not available - training will be slow on CPU")
        recommendations.append("Use Kaggle GPU or Google Colab for faster training")
    
    # Check configuration
    if config.NUM_FRAMES < 20:
        issues.append(f"NUM_FRAMES={config.NUM_FRAMES} might be too low")
        recommendations.append("Consider using 30+ frames for better temporal modeling")
    
    if config.BATCH_SIZE > 16:
        issues.append(f"BATCH_SIZE={config.BATCH_SIZE} might cause OOM on smaller GPUs")
        recommendations.append("Reduce batch size to 8 if you encounter memory errors")
    
    # Print issues
    if issues:
        print("⚠️  POTENTIAL ISSUES:")
        for i, issue in enumerate(issues, 1):
            print(f"  {i}. {issue}")
        print()
    else:
        print("✅ No major issues detected\n")
    
    # Print recommendations
    if recommendations:
        print("💡 RECOMMENDATIONS:")
        for i, rec in enumerate(recommendations, 1):
            print(f"  {i}. {rec}")
        print()
    
    # Check data path
    if not os.path.exists(config.DATA_PATH):
        print(f"❌ DATA PATH NOT FOUND: {config.DATA_PATH}")
        print("   Update config.DATA_PATH to your dataset location")
    else:
        print(f"✅ Data path exists: {config.DATA_PATH}")
    
    print("\n" + "="*60)
    print("📊 EXPECTED PERFORMANCE")
    print("="*60)
    print("With hand detection + augmentation:")
    print("  • Expected accuracy: 75-85%")
    print("  • Training time: 30-60 minutes (GPU)")
    print("  • Memory usage: ~4-6 GB GPU RAM")
    print("\nWithout hand detection:")
    print("  • Expected accuracy: 60-75%")
    print("  • More sensitive to background variations")
    print("="*60 + "\n")

# Run analysis
analyze_code_issues()


🔍 CODE ANALYSIS & RECOMMENDATIONS

⚠️  POTENTIAL ISSUES:
  1. MediaPipe not installed - hand detection disabled
  2. NUM_FRAMES=16 might be too low

💡 RECOMMENDATIONS:
  1. Install MediaPipe: pip install mediapipe
  2. Consider using 30+ frames for better temporal modeling

✅ Data path exists: /kaggle/input/asl-20-words-dataset-v1/Full Data

📊 EXPECTED PERFORMANCE
With hand detection + augmentation:
  • Expected accuracy: 75-85%
  • Training time: 30-60 minutes (GPU)
  • Memory usage: ~4-6 GB GPU RAM

Without hand detection:
  • Expected accuracy: 60-75%
  • More sensitive to background variations

